#03 - Limpeza na Camada Silver - Histórico

---

Importando todas as bibliotecas que serão utilizadas durante o notebook, em seguida definimos todas as variáveis que serão utilizadas durante a execução.

### Variáveis e suas utilizações
historico_bronze_path = define o caminho de leitura da tabela bronze do histórico

historico_silver_path = define o caminho onde será criada a tabela silver do histórico

tickers = informa os tickers que serão processados

---

In [0]:
import pyspark.sql.functions as sf
from pyspark.sql.window import Window

historico_bronze_path = "workspace.stocks.bronze"
historico_silver_path = "workspace.stocks.silver"
tickers = ["PETR4.SA", "VALE3.SA", "BBAS3.SA"]

---

Lemos a tabela bronze filtrando pelas partições dos tickers definidos, evitando leitura desnecessária de dados. Verificamos coluna por coluna a existência de valores nulos, exibindo um resumo antes de removê-los com dropna.

---

In [0]:
df = spark.read.table(historico_bronze_path).filter(sf.col("ticker").isin(tickers))

df_base = df.count()

for c in df.columns:
    df_nulos = df.filter(sf.col(c).isNull()).count()
    if df_nulos > 0:
        print(f"-----{df_nulos} nulos na coluna {c}-----")
    else:
        print(f"-----Sem nulos na coluna {c}-----")

df = df.dropna()

---

Utilizamos uma Window Function particionada por ticker e event_time, ordenada pelo timestamp de ingestão de forma decrescente. Isso garante que em caso de duplicatas, somente o registro mais recente seja mantido.

---

In [0]:
window_spec = Window.partitionBy("ticker", "event_time").orderBy(sf.col("ingestao_ts").desc())
df = (df.withColumn("rn", sf.row_number().over(window_spec))
      .filter(sf.col("rn") == 1)
      .drop("rn")
)

print("-----Removidas duplicidades-----")

---

Adicionamos colunas calculadas para enriquecer os dados históricos. A coluna week_year identifica a semana e ano de cada registro. As colunas variacao_real e variacao_percent calculam a variação do dia em reais e em percentual. 

Por fim, criamos uma flag para identificar registros com valores inválidos nos preços ou volume, removendo-os em seguida.

---

In [0]:
df = (df.withColumn("week_year", sf.concat(sf.weekofyear("event_time"), sf.lit("-"), sf.year("event_time")))
      .withColumn("variacao_real", (sf.col("close") - sf.col("open")))
      .withColumn("variacao_percent", (sf.col("variacao_real") / sf.col("open")*100))
      .withColumn("flag_valor_invalido",
                   sf.when(
                           (sf.col("open") <= 0) |
                           (sf.col("high") <= 0) |
                           (sf.col("low") <= 0) |
                           (sf.col("close") <= 0) |
                           (sf.col("volume") < 0), 
                           True
                   ).otherwise(False)
                   ))

df = df.filter(sf.col("flag_valor_invalido") == False)
df = df.drop("flag_valor_invalido")

print("-----Tabela limpa e enriquecida-----")

---

Exibimos o resumo de linhas removidas durante a limpeza, salvamos o dataframe tratado na tabela silver no formato Delta com particionamento por ticker e exibimos uma amostra do resultado final.

---

In [0]:
df_limpo = df.count()
print(f"-----{df_base} -> {df_limpo}, foram removidas {df_base - df_limpo} linhas-----")

(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("ticker")
    .option("overwriteSchema", "true")
    .saveAsTable(historico_silver_path)
)

display(df)
df.printSchema()